<a href="https://colab.research.google.com/github/iking919/Detecting_Financial_Fraud_via_GNNs/blob/tuning-GNN/notebooks/04_GNN_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GNN Modeling

### Data Extraction
First, we mount Google Drive to access our dataset and extract the processed graph data from a ZIP archive so we can use it locally in the Colab instance.

In [37]:
from google.colab import drive
import zipfile

# mount drive
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/GNN_fraud_Project/processed_data.zip'
extract_path = '/content'

# unzip the processed data locally to speed things up
with zipfile.ZipFile(drive_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Data Inspection
We load and inspect the first few rows of the extracted CSV files to ensure all data was extracted correctly.

In [38]:
import os
import pandas as pd

extract_path = '/content/data/processed'
csv_files = [f for f in os.listdir(extract_path) if f.endswith('.csv')]

for file in csv_files:
    file_path = os.path.join(extract_path, file)

    # Read just the first few rows to quickly inspect columns and types
    df = pd.read_csv(file_path, nrows=5)
    print(f"\n{'='*40}\nDataset: {file}\n{'='*40}")
    print(f"Number of features: {len(df.columns)}")
    print("Features:", list(df.columns))
    display(df.head(2))


Dataset: elliptic_nodes.csv
Number of features: 168
Features: ['timestep', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62', 'feature_63', 'feature_64', 'feature_65', 'feature_66', 'feature_

,timestep,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,tx_id,label
0,-1.556878,0.307025,1.168821,-0.748109,7.643807,-0.053406,6.039024,21.528480,-0.146334,-0.084347,...,0.155526,0.155844,2.115930,0.016322,-0.117599,0.593442,-0.08332,-0.082052,NaN,2
1,-1.556878,0.067780,0.271911,-0.226081,2.628539,-0.053406,2.820415,1.501088,-0.146334,-0.127352,...,0.155526,0.155844,1.200416,-0.028141,-0.117599,0.270020,-0.08332,-0.082052,NaN,2



Dataset: elliptic_edges.csv
Number of features: 2
Features: ['source', 'target']


,source,target
0,230425980,5530458
1,232022460,232438397



Dataset: ieee_train_nodes.csv
Number of features: 434
Features: ['TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 

...

,source,target
0,36634,164336
1,164336,223739



Dataset: paysim_edges.csv
Number of features: 2
Features: ['source', 'target']


,source,target
0,95,2192036
1,834,1359401



Dataset: ieee_test_edges.csv
Number of features: 2
Features: ['source', 'target']


,source,target
0,420980,449477
1,381391,393300


### Environment Setup
We need to install the `torch_geometric` library, which provides the necessary modules for building and training our Graph Neural Networks.

In [39]:
!pip install torch_geometric

### Library Imports and Seed Initialization
Here we import necessary libraries such as PyTorch, PyTorch Geometric, Pandas, and Scikit-Learn. We also set universal random seeds across NumPy and PyTorch to ensure our experiments are reproducible.

In [40]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Set seed for reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


## GraphSAGE (Graph Sample and Aggregate) Model

### GraphSAGE Definition and Utilities
Below, we define the `FraudGraphSAGE` model architecture. Additionally, we establish critical utility functions:
*   `load_graph_data`: Handles loading nodes/edges, preprocessing features (imputation, scaling), splitting the dataset, and dynamically calculating base positive weights to combat class imbalance.
*   `train_gnn` & `evaluate_gnn`: Standardized training and evaluation loops with early stopping.

In [41]:
from sklearn.metrics import average_precision_score

class FraudGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(FraudGraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        # 50% dropout seems to work best
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

def load_graph_data(nodes_filepath, edges_filepath, target_col):
    print(f"Loading graph data...")
    nodes_df = pd.read_csv(nodes_filepath)
    edges_df = pd.read_csv(edges_filepath)

    # clean up labeling
    label_mapping = {'1': 1, 1: 1, '2': 0, 2: 0, 'unknown': -1, '3': -1, 3: -1, '0': 0, 0: 0}
    nodes_df[target_col] = nodes_df[target_col].map(label_mapping)

    node_id_col = 'tx_id' if 'tx_id' in nodes_df.columns else ('TransactionID' if 'TransactionID' in nodes_df.columns else None)

    if node_id_col:
        # drop dupes just in case
        if nodes_df[node_id_col].duplicated().any():
            print(f"Warning: Duplicate node IDs found in '{node_id_col}'. Keeping first.")
            nodes_df = nodes_df.drop_duplicates(subset=[node_id_col], keep='first')

        original_node_ids = nodes_df[node_id_col].values
        node_id_to_idx = {id: i for i, id in enumerate(original_node_ids)}

        # remap edges to 0-indexed ints
        edges_df['source'] = edges_df['source'].map(node_id_to_idx)
        edges_df['target'] = edges_df['target'].map(node_id_to_idx)
        edges_df.dropna(subset=['source', 'target'], inplace=True)

        features_to_drop = [target_col, node_id_col]
        X = nodes_df.drop(columns=features_to_drop, errors='ignore').select_dtypes(include=[np.number]).values
    else:
        X = nodes_df.drop(columns=[target_col], errors='ignore').select_dtypes(include=[np.number]).values

    y = nodes_df[target_col].values

    # basic imputation and scaling
    imputer = SimpleImputer(strategy='mean')
    X_imputed = imputer.fit_transform(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    x_tensor = torch.tensor(X_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.float)

    source_nodes = edges_df['source'].astype(int).values
    target_nodes = edges_df['target'].astype(int).values
    edge_index = torch.tensor(np.vstack((source_nodes, target_nodes)), dtype=torch.long)

    data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

    # only split on nodes we actually have labels for
    known_indices = np.where((y == 0) | (y == 1))[0]

    if len(np.unique(y[known_indices])) < 2:
        train_indices, test_indices = train_test_split(known_indices, test_size=0.2, random_state=42)
    else:
        train_indices, test_indices = train_test_split(known_indices, test_size=0.2, stratify=y[known_indices], random_state=42)

    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    train_mask[train_indices] = True
    test_mask[test_indices] = True
    data.train_mask = train_mask
    data.test_mask = test_mask

    # initial weight calc (usually gets overridden in sweeps)
    num_pos = int((y_tensor[train_mask] == 1).sum().item())
    num_neg = int((y_tensor[train_mask] == 0).sum().item())
    raw_weight = num_neg / max(1, num_pos)
    capped_weight = min(10.0, raw_weight)
    pos_weight = torch.tensor([capped_weight], dtype=torch.float)

    return data, pos_weight

def train_gnn(model, data, pos_weight, device, epochs=300, lr=0.01, patience=30):
    model.to(device)
    data = data.to(device)
    pos_weight = pos_weight.to(device)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index).squeeze()
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # tracking for early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:03d}/{epochs} | Loss: {loss:.4f}')

        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1} (Loss: {loss:.4f})')
            break

def evaluate_gnn(model, data, device):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        out = model(data.x, data.edge_index).squeeze()
        probs = torch.sigmoid(out)
        test_probs = probs[data.test_mask].cpu().numpy()
        test_y = data.y[data.test_mask].cpu().numpy()

    # pull standard metrics
    auc_roc = roc_auc_score(test_y, test_probs) if len(np.unique(test_y)) > 1 else 0.0
    auc_pr = average_precision_score(test_y, test_probs) if len(np.unique(test_y)) > 1 else 0.0
    print(f"AUC-ROC: {auc_roc:.4f} | AUC-PR: {auc_pr:.4f}")


## GCN (Graph Convolutional Network) Model

We define the `FraudGCN` model architecture, utilizing standard Graph Convolutional layers (`GCNConv`) with dropout for regularization.

In [42]:
class FraudGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(FraudGCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

## GAT (Graph Attention Networks) Model

We define the `FraudGAT` model. This model uses multi-head graph attention mechanisms (`GATConv`) to weigh neighboring nodes differently based on their features.

In [43]:
class FraudGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super(FraudGAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels // heads, heads=heads)
        self.conv2 = GATConv(hidden_channels, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

## Threshold Tuning Strategy
Since fraud detection involves highly imbalanced classes, standard evaluation metrics like Accuracy are misleading. The function below evaluates our models by sweeping across different probability thresholds to maximize the F1-score specifically for the minority (fraud) class.

In [47]:
def tune_model_threshold(model, data, dataset_name, weight):
    print(f"\n{'-'*15} Tuning Thresholds: {dataset_name} (Weight: {weight}) {'-'*15}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()
    model.to(device)
    data = data.to(device)

    with torch.no_grad():
        out = model(data.x, data.edge_index).squeeze()
        probs = torch.sigmoid(out[data.test_mask]).cpu().numpy()
        y_true = data.y[data.test_mask].cpu().numpy()

    auc_pr = average_precision_score(y_true, probs) if len(np.unique(y_true)) > 1 else 0.0
    print(f"AUC-PR: {auc_pr:.4f}")

    thresholds = np.linspace(0.01, 0.99, 50)
    best_f1 = 0
    best_metrics = {}

    for t in thresholds:
        preds = (probs > t).astype(float)
        report = classification_report(y_true, preds, output_dict=True, zero_division=0)
        f1_pos = report['1.0']['f1-score']

        if f1_pos > best_f1:
            best_f1 = f1_pos
            best_metrics = {
                'AUC-PR': auc_pr,
                'Threshold': t,
                'Weight': weight,
                'Accuracy': report['accuracy'],
                'Recall': report['1.0']['recall'],
                'F1 (Macro)': report['macro avg']['f1-score'],
                'F1 (Weighted)': report['weighted avg']['f1-score'],
                'Precision': report['1.0']['precision'],
                'F1 (Pos)': f1_pos
            }

    if best_metrics:
        print(f"> Best Threshold: {best_metrics['Threshold']:.2f} | Best F1 (Pos): {best_f1:.4f}")
        print(f"> Precision: {best_metrics['Precision']:.4f} | Recall: {best_metrics['Recall']:.4f} | F1 (Macro): {best_metrics['F1 (Macro)']:.4f}")

    return best_f1, best_metrics


## Pipeline Executions

### GraphSAGE Pipeline
We now execute the end-to-end training and tuning pipeline for GraphSAGE across all three datasets (PaySim, IEEE-CIS, and Elliptic). For each dataset, we perform a search over `pos_weight` values to find the best configuration.

In [48]:
def run_sage_pipeline(dataset_name, nodes_path, edges_path, target_col):
    print(f"\n{'='*40}\n Processing {dataset_name} \n{'='*40}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load data once
    data, _ = load_graph_data(nodes_path, edges_path, target_col)

    best_overall_f1 = 0
    best_model = None
    best_model_metrics = None

    # Grid search for pos_weight
    for w in [5, 10, 20, 50]:
        print(f"\n--- Training GraphSAGE with pos_weight = {w} ---")
        model = FraudGraphSAGE(data.num_features, 128, 1)
        weight_tensor = torch.tensor([w], dtype=torch.float)

        train_gnn(model, data, weight_tensor, device, epochs=300, patience=20)
        f1, metrics = tune_model_threshold(model, data, dataset_name, w)

        if f1 > best_overall_f1:
            best_overall_f1 = f1
            best_model = model
            best_model_metrics = metrics

    print(f"\n{'='*40}")
    print(f"BEST {dataset_name} GraphSAGE MODEL RESULTS:")
    if best_model_metrics:
        print(f"Weight: {best_model_metrics['Weight']}")
        print(f"Threshold: {best_model_metrics['Threshold']:.2f}")
        print(f"AUC-PR: {best_model_metrics['AUC-PR']:.4f}")
        print(f"Accuracy: {best_model_metrics['Accuracy']:.4f}")
        print(f"Recall: {best_model_metrics['Recall']:.4f}")
        print(f"F1 (Pos): {best_model_metrics['F1 (Pos)']:.4f}")
        print(f"F1 (Macro): {best_model_metrics['F1 (Macro)']:.4f}")
        print(f"F1 (Weighted): {best_model_metrics['F1 (Weighted)']:.4f}")
    print(f"{'='*40}")
    return best_model

# Retrain GraphSAGE for all three datasets with pos_weight tuning
paysim_sage_model = run_sage_pipeline(
    "PaySim",
    "/content/data/processed/paysim_nodes.csv",
    "/content/data/processed/paysim_edges.csv",
    "label"
)

ieee_sage_model = run_sage_pipeline(
    "IEEE-CIS",
    "/content/data/processed/ieee_train_nodes.csv",
    "/content/data/processed/ieee_train_edges.csv",
    "label"
)

elliptic_sage_model = run_sage_pipeline(
    "Elliptic",
    "/content/data/processed/elliptic_nodes.csv",
    "/content/data/processed/elliptic_edges.csv",
    "label"
)



 Processing PaySim 
Loading graph data...

--- Training GraphSAGE with pos_weight = 5 ---
Epoch 050/300 | Loss: 0.0366
Epoch 100/300 | Loss: 0.0309
Epoch 150/300 | Loss: 0.0280
Epoch 200/300 | Loss: 0.0265
Epoch 250/300 | Loss: 0.0255
Epoch 300/300 | Loss: 0.0250

--------------- Tuning Thresholds: PaySim (Weight: 5) ---------------
AUC-PR: 0.4249
> Best Threshold: 0.21 | Best F1 (Pos): 0.4806
> Precision: 0.7313 | Recall: 0.3579 | F1 (Macro): 0.7400

--- Training GraphSAGE with pos_weight = 10 ---
Epoch 050/300 | Loss: 0.0580
Epoch 100/300 | Loss: 0.0486
Epoch 150/300 | Loss: 0.0442
Epoch 200/300 | Loss: 0.0416
Epoch 250/300 | Loss: 0.0399
Epoch 300/300 | Loss: 0.0387

--------------- Tuning Thresholds: PaySim (Weight: 10) ---------------
AUC-PR: 0.4890
> Best Threshold: 0.33 | Best F1 (Pos): 0.5361
> Precision: 0.7691 | Recall: 0.4114 | F1 (Macro): 0.7678

--- Training GraphSAGE with pos_weight = 20 ---
Epoch 050/300 | Loss: 0.0926
Epoch 100/300 | Loss: 0.0745
Epoch 150/300 | Loss: 

### GCN Pipeline
Similarly, we run the evaluation pipeline using the GCN model on all three datasets to compare against the GraphSAGE benchmark.

In [49]:
from torch_geometric.nn import GCNConv

def run_gcn_pipeline(dataset_name, nodes_path, edges_path, target_col):
    print(f"\n{'='*40}\n Processing {dataset_name} (GCN)\n{'='*40}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load data once
    data, _ = load_graph_data(nodes_path, edges_path, target_col)

    best_overall_f1 = 0
    best_model = None
    best_model_metrics = None

    # Grid search for pos_weight
    for w in [5, 10, 20, 50]:
        print(f"\n--- Training GCN with pos_weight = {w} ---")
        model = FraudGCN(data.num_features, 128, 1)
        weight_tensor = torch.tensor([w], dtype=torch.float)

        train_gnn(model, data, weight_tensor, device, epochs=300, patience=20)
        f1, metrics = tune_model_threshold(model, data, f"{dataset_name} (GCN)", w)

        if f1 > best_overall_f1:
            best_overall_f1 = f1
            best_model = model
            best_model_metrics = metrics

    print(f"\n{'='*40}")
    print(f"BEST {dataset_name} GCN MODEL RESULTS:")
    if best_model_metrics:
        print(f"Weight: {best_model_metrics['Weight']}")
        print(f"Threshold: {best_model_metrics['Threshold']:.2f}")
        print(f"AUC-PR: {best_model_metrics['AUC-PR']:.4f}")
        print(f"Accuracy: {best_model_metrics['Accuracy']:.4f}")
        print(f"Recall: {best_model_metrics['Recall']:.4f}")
        print(f"F1 (Pos): {best_model_metrics['F1 (Pos)']:.4f}")
        print(f"F1 (Macro): {best_model_metrics['F1 (Macro)']:.4f}")
        print(f"F1 (Weighted): {best_model_metrics['F1 (Weighted)']:.4f}")
    print(f"{'='*40}")
    return best_model

# Retrain GCN for all three datasets with pos_weight tuning
paysim_gcn_model = run_gcn_pipeline(
    "PaySim",
    "/content/data/processed/paysim_nodes.csv",
    "/content/data/processed/paysim_edges.csv",
    "label"
)

ieee_gcn_model = run_gcn_pipeline(
    "IEEE-CIS",
    "/content/data/processed/ieee_train_nodes.csv",
    "/content/data/processed/ieee_train_edges.csv",
    "label"
)

elliptic_gcn_model = run_gcn_pipeline(
    "Elliptic",
    "/content/data/processed/elliptic_nodes.csv",
    "/content/data/processed/elliptic_edges.csv",
    "label"
)



 Processing PaySim (GCN)
Loading graph data...

--- Training GCN with pos_weight = 5 ---
Epoch 050/300 | Loss: 0.0382
Epoch 100/300 | Loss: 0.0336
Epoch 150/300 | Loss: 0.0320
Epoch 200/300 | Loss: 0.0309
Epoch 250/300 | Loss: 0.0303
Epoch 300/300 | Loss: 0.0298

--------------- Tuning Thresholds: PaySim (GCN) (Weight: 5) ---------------
AUC-PR: 0.1757
> Best Threshold: 0.27 | Best F1 (Pos): 0.2604
> Precision: 0.3660 | Recall: 0.2021 | F1 (Macro): 0.6298

--- Training GCN with pos_weight = 10 ---
Epoch 050/300 | Loss: 0.0597
Epoch 100/300 | Loss: 0.0546
Epoch 150/300 | Loss: 0.0522
Epoch 200/300 | Loss: 0.0509
Epoch 250/300 | Loss: 0.0501
Epoch 300/300 | Loss: 0.0496

--------------- Tuning Thresholds: PaySim (GCN) (Weight: 10) ---------------
AUC-PR: 0.2153
> Best Threshold: 0.45 | Best F1 (Pos): 0.2933
> Precision: 0.4123 | Recall: 0.2276 | F1 (Macro): 0.6463

--- Training GCN with pos_weight = 20 ---
Epoch 050/300 | Loss: 0.0971
Epoch 100/300 | Loss: 0.0907
Epoch 150/300 | Loss: 0

### GAT Pipeline
Finally, we evaluate the Graph Attention Network (GAT) model across the datasets, searching for the optimal class weight and threshold combination.

In [59]:
import gc
import copy
from torch_geometric.nn import GATConv

def run_gat_pipeline(dataset_name, nodes_path, edges_path, target_col):
    print(f"\n{'='*40}\n Processing {dataset_name} (GAT)\n{'='*40}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load data once
    data, _ = load_graph_data(nodes_path, edges_path, target_col)

    best_overall_f1 = 0
    best_model = None
    best_model_metrics = None

    # Grid search for pos_weight
    for w in [5, 10, 20, 50]:
        print(f"\n--- Training GAT with pos_weight = {w} ---")
        model = FraudGAT(data.num_features, 32, 1, heads=2)
        weight_tensor = torch.tensor([w], dtype=torch.float)

        train_gnn(model, data, weight_tensor, device, epochs=300, patience=20)
        f1, metrics = tune_model_threshold(model, data, f"{dataset_name} (GAT)", w)

        if f1 > best_overall_f1:
            best_overall_f1 = f1
            # Move the best model to CPU to free up GPU memory
            best_model = copy.deepcopy(model).cpu()
            best_model_metrics = metrics

        # Aggressive memory cleanup after every single weight iteration
        del model
        del weight_tensor
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\n{'='*40}")
    print(f"BEST {dataset_name} GAT MODEL RESULTS:")
    if best_model_metrics:
        print(f"Weight: {best_model_metrics['Weight']}")
        print(f"Threshold: {best_model_metrics['Threshold']:.2f}")
        print(f"AUC-PR: {best_model_metrics['AUC-PR']:.4f}")
        print(f"Accuracy: {best_model_metrics['Accuracy']:.4f}")
        print(f"Recall: {best_model_metrics['Recall']:.4f}")
        print(f"F1 (Pos): {best_model_metrics['F1 (Pos)']:.4f}")
        print(f"F1 (Macro): {best_model_metrics['F1 (Macro)']:.4f}")
        print(f"F1 (Weighted): {best_model_metrics['F1 (Weighted)']:.4f}")
    print(f"{'='*40}")

    # Cleanup dataset memory before moving to the next dataset
    del data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_model

# Retrain GAT for all three datasets with pos_weight tuning
paysim_gat_model = run_gat_pipeline(
    "PaySim",
    "/content/data/processed/paysim_nodes.csv",
    "/content/data/processed/paysim_edges.csv",
    "label"
)

ieee_gat_model = run_gat_pipeline(
    "IEEE-CIS",
    "/content/data/processed/ieee_train_nodes.csv",
    "/content/data/processed/ieee_train_edges.csv",
    "label"
)

elliptic_gat_model = run_gat_pipeline(
    "Elliptic",
    "/content/data/processed/elliptic_nodes.csv",
    "/content/data/processed/elliptic_edges.csv",
    "label"
)



 Processing PaySim (GAT)
Loading graph data...

--- Training GAT with pos_weight = 5 ---
Epoch 050/300 | Loss: 0.0497
Epoch 100/300 | Loss: 0.0396
Epoch 150/300 | Loss: 0.0351
Epoch 200/300 | Loss: 0.0329
Epoch 250/300 | Loss: 0.0317
Epoch 300/300 | Loss: 0.0308

--------------- Tuning Thresholds: PaySim (GAT) (Weight: 5) ---------------
AUC-PR: 0.0831
> Best Threshold: 0.43 | Best F1 (Pos): 0.1996
> Precision: 0.2396 | Recall: 0.1710 | F1 (Macro): 0.5993

--- Training GAT with pos_weight = 10 ---
Epoch 050/300 | Loss: 0.0673
Epoch 100/300 | Loss: 0.0587
Epoch 150/300 | Loss: 0.0547
Epoch 200/300 | Loss: 0.0518
Epoch 250/300 | Loss: 0.0499
Epoch 300/300 | Loss: 0.0485

--------------- Tuning Thresholds: PaySim (GAT) (Weight: 10) ---------------
AUC-PR: 0.1659
> Best Threshold: 0.57 | Best F1 (Pos): 0.2556
> Precision: 0.2875 | Recall: 0.2301 | F1 (Macro): 0.6274

--- Training GAT with pos_weight = 20 ---
Epoch 050/300 | Loss: 0.1052
Epoch 100/300 | Loss: 0.0915
Epoch 150/300 | Loss: 0

In [58]:
import gc
import torch

print("Starting aggressive garbage collection...")

# Run garbage collection multiple times to catch circular references
for _ in range(3):
    gc.collect()

# Empty PyTorch's CUDA cache and IPC to free up GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("\n--- GPU Memory Status ---")
    print(torch.cuda.memory_summary(abbreviated=True))
    print("CUDA cache and IPC cleared.")

print("Aggressive garbage collection complete. Ready for GAT pipeline!")

Starting aggressive garbage collection...

--- GPU Memory Status ---
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 4            |        cudaMalloc retries: 6         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7682 MiB |  89094 MiB | 643151 GiB | 643144 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   7682 MiB |  89094 MiB | 643151 GiB | 643144 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |   7682 MiB |  89093 MiB | 643120 GiB | 643113 G